In [1]:
import os

data_dir = "/kaggle/input/datasets/ibrahimhany202200518/gec-c4-5m-splits/data"
for f in os.listdir(data_dir):
    size_mb = os.path.getsize(os.path.join(data_dir, f)) / 1e6
    print(f"  {f} → {size_mb:.1f} MB")

  test.tsv → 21.2 MB
  train.tsv → 1184.8 MB
  val.tsv → 41.3 MB


## 1. Load training data and build classification samples

Each (corrupted, clean) pair becomes two samples:
- corrupted sentence → label 0 (ungrammatical)
- clean sentence → label 1 (grammatical)

In [2]:
import time

TRAIN_PATH = "/kaggle/input/datasets/ibrahimhany202200518/gec-c4-5m-splits/data/train.tsv"

texts, labels = [], []

start = time.time()
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            corrupted, clean = parts
            texts.append(corrupted)
            labels.append(0)
            texts.append(clean)
            labels.append(1)

elapsed = time.time() - start
print(f"Total samples   : {len(texts):,}")
print(f"Label 0 (bad)   : {labels.count(0):,}")
print(f"Label 1 (good)  : {labels.count(1):,}")
print(f"Load time       : {elapsed:.1f}s")
print(f"\nSample preview:")
for i in [0, 1, 4, 5]:
    print(f"  [{labels[i]}] {texts[i][:80]}")

Total samples   : 9,500,000
Label 0 (bad)   : 4,750,000
Label 1 (good)  : 4,750,000
Load time       : 27.1s

Sample preview:
  [0] He presented theresults at the american society of Human Genetics 2015 Annual Me
  [1] He presented the results at the American Society of Human Genetics 2015 Annual M
  [0] The new 15 percents tax bracket kicks in and applies to incomes above the 0 perc
  [1] The new 15 percent tax bracket kicks in and applies to incomes above the 0-perce


## 2. Load validation and test sets

In [3]:
VAL_PATH  = "/kaggle/input/datasets/ibrahimhany202200518/gec-c4-5m-splits/data/val.tsv"
TEST_PATH = "/kaggle/input/datasets/ibrahimhany202200518/gec-c4-5m-splits/data/test.tsv"

def load_split(path):
    texts, labels = [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                corrupted, clean = parts
                texts.append(corrupted)
                labels.append(0)
                texts.append(clean)
                labels.append(1)
    return texts, labels

val_texts,  val_labels  = load_split(VAL_PATH)
test_texts, test_labels = load_split(TEST_PATH)

print(f"Val  samples : {len(val_texts):,}")
print(f"Test samples : {len(test_texts):,}")

Val  samples : 330,000
Test samples : 170,000


## 3. Build TF-IDF features

We vectorize all three splits using TF-IDF. The vectorizer is fit **only on training data** — val and test are just transformed. This is the correct ML practice: no data leakage.

We cap the vocabulary at 50,000 terms and use unigrams + bigrams. Bigrams help capture some local order (e.g. "go to" vs "went to") even within a BoW framework.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("Fitting TF-IDF vectorizer on training data...")
start = time.time()

vectorizer = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),   # unigrams + bigrams
    sublinear_tf=True,    # apply log(tf) + 1 to compress large counts
    min_df=3,             # ignore terms appearing in fewer than 3 docs
)

X_train = vectorizer.fit_transform(texts)
X_val   = vectorizer.transform(val_texts)
X_test  = vectorizer.transform(test_texts)

y_train = labels
y_val   = val_labels
y_test  = test_labels

elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s")
print(f"Vocabulary size : {len(vectorizer.vocabulary_):,}")
print(f"X_train shape   : {X_train.shape}")
print(f"X_val shape     : {X_val.shape}")
print(f"X_test shape    : {X_test.shape}")

Fitting TF-IDF vectorizer on training data...
Done in 520.5s
Vocabulary size : 50,000
X_train shape   : (9500000, 50000)
X_val shape     : (330000, 50000)
X_test shape    : (170000, 50000)


## 4. Train Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression

print("Training Logistic Regression...")
start = time.time()

lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="saga",      # best solver for large sparse data
    n_jobs=-1,
    random_state=42,
)
lr_model.fit(X_train, y_train)

elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s")

Training Logistic Regression...
Done in 245.9s


## 5. Train Linear SVM

In [6]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

print("Training Linear SVM...")
start = time.time()

svm_base = LinearSVC(C=0.1, max_iter=2000, random_state=42)
svm_model = CalibratedClassifierCV(svm_base, cv=3)
svm_model.fit(X_train, y_train)

elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s")

Training Linear SVM...
Done in 1158.0s


## 6. Evaluate all models

In [7]:
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)
import numpy as np

def evaluate(model, X, y, split_name, model_name):
    preds = model.predict(X)
    acc   = accuracy_score(y, preds)
    cm    = confusion_matrix(y, preds)
    report = classification_report(y, preds, target_names=["ungrammatical", "grammatical"])
    
    print(f"\n{'='*50}")
    print(f"{model_name} — {split_name}")
    print(f"{'='*50}")
    print(f"Accuracy : {acc:.4f}")
    print(report)
    print(f"Confusion matrix:\n{cm}")
    return acc, preds

lr_val_acc,  lr_val_preds  = evaluate(lr_model,  X_val,  y_val,  "Validation", "Logistic Regression")
lr_test_acc, lr_test_preds = evaluate(lr_model,  X_test, y_test, "Test",       "Logistic Regression")

svm_val_acc,  svm_val_preds  = evaluate(svm_model, X_val,  y_val,  "Validation", "Linear SVM")
svm_test_acc, svm_test_preds = evaluate(svm_model, X_test, y_test, "Test",       "Linear SVM")


Logistic Regression — Validation
Accuracy : 0.6226
               precision    recall  f1-score   support

ungrammatical       0.63      0.58      0.61    165000
  grammatical       0.61      0.67      0.64    165000

     accuracy                           0.62    330000
    macro avg       0.62      0.62      0.62    330000
 weighted avg       0.62      0.62      0.62    330000

Confusion matrix:
[[ 95505  69495]
 [ 55057 109943]]

Logistic Regression — Test
Accuracy : 0.6229
               precision    recall  f1-score   support

ungrammatical       0.63      0.58      0.61     85000
  grammatical       0.61      0.66      0.64     85000

     accuracy                           0.62    170000
    macro avg       0.62      0.62      0.62    170000
 weighted avg       0.62      0.62      0.62    170000

Confusion matrix:
[[49411 35589]
 [28524 56476]]

Linear SVM — Validation
Accuracy : 0.6224
               precision    recall  f1-score   support

ungrammatical       0.63      0.58 

In [11]:
print("="*50)
print("MISCLASSIFIED EXAMPLES (Logistic Regression)")
print("="*50)

test_sentences = []
with open("/kaggle/input/datasets/ibrahimhany202200518/gec-c4-5m-splits/data/test.tsv", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            test_sentences.append((parts[0], parts[1]))
        if len(test_sentences) >= 500:
            break

count = 0
for i, (corrupted, clean) in enumerate(test_sentences[:200]):
    # Check corrupted sentence (label=0, index 2*i)
    idx = 2 * i
    true_label = y_test[idx]
    pred_label = lr_model.predict(X_test[idx])[0]
    
    if true_label != pred_label:
        print(f"❌ Corrupted misclassified as grammatical:")
        print(f"   IN : {corrupted[:80]}")
        print(f"   OUT: {clean[:80]}")
        print()
        count += 1
    if count >= 5:
        break

MISCLASSIFIED EXAMPLES (Logistic Regression)
❌ Corrupted misclassified as grammatical:
   IN : Silicon Valley companies and investors have informed that the Internet startups 
   OUT: Silicon Valley companies and investors suggest that the Internet start-ups they 

❌ Corrupted misclassified as grammatical:
   IN : Looking for an international phone card for overseas call from China to Romania 
   OUT: Looking for an international phone card for overseas calling from China to Roman

❌ Corrupted misclassified as grammatical:
   IN : Now the three Trusts will begin 12-week consultation with staff and other stakeh
   OUT: Now the three Trusts will begin a 12-week consultation with staff and other stak

❌ Corrupted misclassified as grammatical:
   IN : When it came to Sandy Hook, an old video was shown, with Jones saying " It took 
   OUT: When it came to Sandy Hook, an old video was shown, with Jones saying, "It took 

❌ Corrupted misclassified as grammatical:
   IN : Before it was banned,